In [13]:
import pandas as pd
import re
import unicodedata

In [14]:
def normalize_text(text):
    """
    Normalize text for comparison:
    - Uppercase
    - Remove accents
    - Remove punctuation
    - Collapse multiple spaces
    """
    if pd.isna(text):
        return ""

    text = str(text).upper()

    # Remove accents
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))

    # Replace punctuation with spaces
    text = re.sub(r"[^A-Z0-9]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def has_location(customer_name, city, state, country):
    """
    Returns:
        True  -> Customer name already contains city, state or country
        False -> No location found
    """

    customer = normalize_text(customer_name)

    locations = [
        normalize_text(city),
        normalize_text(state),
        normalize_text(country)
    ]

    for loc in locations:
        if loc != "" and loc in customer:
            return True

    return False


def suggest_customer_name(customer_name, city):
    """
    Appends the city only if it is not already present.
    """

    if pd.isna(city) or str(city).strip() == "":
        return customer_name

    return f"{customer_name} {city}".strip()

In [15]:
df = pd.read_csv(r'C:\Users\Ivan.DeLaFuente\OneDrive - Quaker Houghton\Documents\QH WorkDay\Junio 2026\T&P Market\T&P_Market.csv')

In [16]:
df["Has Location"] = df.apply(
    lambda row: has_location(
        row["Customer Name"],
        row["Ship To City"],
        row["Ship To State"],
        row["Ship To Country"]
    ),
    axis=1
)

In [17]:
df["Suggested Customer Name"] = df.apply(
    lambda row: (
        row["Customer Name"]
        if row["Has Location"]
        else suggest_customer_name(
            row["Customer Name"],
            row["Ship To City"]
        )
    ),
    axis=1
)

In [18]:
df = df[["Customer Name", "Suggested Customer Name", "Ship To City", "Ship To State", "Ship To Country", "Has Location", ]]
df = df.drop_duplicates(subset=["Customer Name", "Suggested Customer Name"])